# Load System Data and Compute Average Execution Accuracy (EX) per System

In [3]:
import pandas as pd

# Read the System  data
df = pd.read_csv('/Users/mhmalekpour/PycharmProjects/text-to-sql-coverage/data/evaluation/experiments/systems_evel_on_bird/systems_data_with_ex.csv')

# Compute average EX per system
avg_ex_per_system = df.groupby('system')['EX'].mean().reset_index()

print(avg_ex_per_system)

    system        EX
0    chess  0.650587
1  din-sql  0.588657
2  mac-sql  0.559035


# Filter Rows with EX == 1 and Technique Failures (EXP or EXR != 1)

In [6]:
# filters rows where the EX value in EXECUTION_ACCURACY is 1, and then further filters those rows to keep only ones where at least one technique column has EXP or EXR not equal to 1.

import pandas as pd
import json

# Read the CSV
df = pd.read_csv('/Users/mhmalekpour/PycharmProjects/text-to-sql-coverage/data/evaluation/experiments/systems_evel_on_bird/systems_data_with_metrics.csv')

# Parse the JSON columns
technique_cols = [
    'EXACT_COLUMN_AND_EXACT_CELL',
    'SEMANTIC_COLUMN_AND_EXACT_CELL',
    'UNIFIED_COLUMN_AND_SEMANTIC_ROW'
]
for col in ['EXECUTION_ACCURACY'] + technique_cols:
    df[col] = df[col].apply(json.loads)

# Filter EX == 1
filtered = df[df['EXECUTION_ACCURACY'].apply(lambda x: x.get('EX', 0) == 1)]

# Check if any EXP or EXR != 1 in any technique column
def exp_exr_not_1(row):
    for col in technique_cols:
        exp = row[col].get('EXP')
        exr = row[col].get('EXR')
        if exp != 1 or exr != 1:
            return True
    return False

result = filtered[filtered.apply(exp_exr_not_1, axis=1)]

print(len(result))

result.to_csv('/Users/mhmalekpour/PycharmProjects/text-to-sql-coverage/data/evaluation/experiments/systems_evel_on_bird/filtered_results.csv', index=False)

32


# Find Intersection of Failed Questions (EX == 0) Across All Systems

show case sample in paper: why exact column and exact cell is good
chess	california_schools	87	What are the valid e-mail addresses of the administrator of the school located in the San Bernardino county, City of San Bernardino City Unified that opened between 1/1/2009 to 12/31/2010 whose school types are public Intermediate/Middle Schools and Unified Schools?	SELECT T2.AdmEmail1, T2.AdmEmail2 FROM frpm AS T1 INNER JOIN schools AS T2 ON T1.CDSCode = T2.CDSCode WHERE T2.County = 'San Bernardino' AND T2.City = 'San Bernardino' AND T2.DOC = 54 AND strftime('%Y', T2.OpenDate) BETWEEN '2009' AND '2010' AND T2.SOC = 62	SELECT DISTINCT AdmEmail1, AdmEmail2, AdmEmail3  FROM schools  WHERE County = 'San Bernardino'  AND District = 'San Bernardino City Unified'  AND SOC = 62  AND DOC = 54  AND OpenDate BETWEEN '2009-01-01' AND '2010-12-31'  AND (AdmEmail1 IS NOT NULL OR AdmEmail2 IS NOT NULL OR AdmEmail3 IS NOT NULL)	{'EX': 0}	{"EX": 0, "EXP": 1.0, "EXR": 1.0, "F1": 1.0}	{"EX": 0, "EXP": 0.6666666666666666, "EXR": 1.0, "F1": 0.8}	{"EX": 0, "EXP": 0.9503985047340393, "EXR": 0.9503985047340393, "F1": 0.9503985047340393}

In [3]:
# ilters rows where the execution accuracy (EX) is 0 (indicating failure), finds the set of question_ids that failed in all systems (intersection)

import pandas as pd
import json

# Read the CSV
df = pd.read_csv('/Users/mhmalekpour/PycharmProjects/text-to-sql-coverage/data/evaluation/experiments/systems_evel_on_bird/systems_data_with_metrics.csv')

# Parse EXECUTION_ACCURACY column
df['EXECUTION_ACCURACY'] = df['EXECUTION_ACCURACY'].apply(json.loads)

# Filter rows where EX == 0
failures = df[df['EXECUTION_ACCURACY'].apply(lambda x: x.get('EX', 0) == 0)]

# Find intersection of question_ids across all systems
systems = failures['system'].unique()
question_ids_per_system = [set(failures[failures['system'] == sys]['question_id']) for sys in systems]
common_question_ids = set.intersection(*question_ids_per_system)

# Filter failures to keep only rows with common question_ids
intersection_failures = failures[failures['question_id'].isin(common_question_ids)]

print(len(intersection_failures))
intersection_failures.to_csv('/Users/mhmalekpour/PycharmProjects/text-to-sql-coverage/data/evaluation/experiments/systems_evel_on_bird/intersection_failures.csv', index=False)

6
